# 第7章 材料モデルと反射・透過

書籍『Sionna RTによる電波伝搬レイトレーシング入門』第7章のコードである。
セルは本文の掲載順に並べてあり、コードセルの中身は本文に印刷したものと同じである。

先頭の「準備」セルだけは本文に印刷していない。本文のコードが前提にしている
import とシーンの読み込みをここで済ませてある。準備セルを実行すれば、以降は
上から順に実行できる。

ただし、本文が説明のために示している断片（自分で作ったシーンのパスや、その
シーンには無い物体名を使う例）はそのままでは動かない。該当するセルの前に注記を
置いてある。

## 準備

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sionna.rt as rt
from sionna.rt import (load_scene, scene as scene_assets,
                       Transmitter, Receiver, PlanarArray,
                       PathSolver, RadioMapSolver, Camera, RadioMaterial)

# 本書の多くの章で使うミュンヘン中心部の都市シーン
scene = load_scene(scene_assets.munich)
scene.frequency = 3.5e9
scene.tx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')
scene.rx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')
scene.add(Transmitter('tx0', position=[-30.0, 38.0, 35.0]))

solver = PathSolver()
rmsolver = RadioMapSolver()

## 7. Sionna RT の `RadioMaterial`

In [ ]:
from sionna.rt import RadioMaterial

concrete = RadioMaterial(name='concrete',
                          relative_permittivity=5.31,
                          conductivity=0.052,
                          thickness=0.3)
scene.add(concrete)

# 物体名はシーンごとに違う。まず scene.objects で名前を確認する
wall = scene.objects['merged-shapes']
wall.radio_material = concrete

## 8. 材料を変更して結果を比較する

In [ ]:
# 比較用: シーン内の物体をまとめて金属化する
metal = RadioMaterial(name='metal',
                      relative_permittivity=1.0,
                      conductivity=1e7,
                      thickness=0.005)
scene.add(metal)
for obj in scene.objects.values():
    obj.radio_material = metal

rm_solver = RadioMapSolver()
rm_metal = rm_solver(scene, max_depth=3, cell_size=[5.0, 5.0])